In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [33]:
import json
import pandas as pd
import warnings

from numpy import random
from collections import defaultdict
from functools import reduce
import optuna

from sklearn.linear_model import LogisticRegression
from model.optimization import ObjectiveSuggestion, SuggestionValueType, create_study, weighted_f1, micro_recall
from model.dataset import get_dataframe
from utils import get_importances

DEFAULT_RANDOM_SEED = 774
random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
seed_list = list(random.random_integers(low=0, high=2**32 - 1, size=20))
warnings.filterwarnings("ignore")

In [34]:
category = "min_tpm_5"
data = get_dataframe(category=category, dataset="train", drop_columns=[])
subtypes = data["subtype"].unique()

In [35]:
def choose_important_genes(scoring: callable):
  random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
  importances_by_sex_subtype = defaultdict(list)

  for sex in ["Male", "Female"]:
    filtered_sex_dataset = data[data["sex"] == sex]
    for subtype in subtypes:
      print(f"Finding importances for {sex} - {subtype}")
      targeted_data = filtered_sex_dataset.copy()
      targeted_data["subtype_target"] = targeted_data["subtype"] == subtype
      X, y = targeted_data.drop(labels=["subtype_target", "subtype", "sex"], axis=1), targeted_data["subtype_target"]

      best_score = 0
      for seed in seed_list:
        random.mtrand._rand.seed(seed)
        study, best_model = create_study(
          model_factory=lambda **kwargs: LogisticRegression(**kwargs, class_weight="balanced", random_state=seed, solver="liblinear", max_iter=100),
          scoring=scoring,
          custom_dataset=(X, y),
          suggestions=[
            ObjectiveSuggestion(
              value_type=SuggestionValueType.CATEGORY,
              param="penalty",
              param_options=["l1", "l2"]
            ),
            ObjectiveSuggestion(
              value_type=SuggestionValueType.FLOAT,
              param="C",
              param_range=(0, 5)
            )
          ],
          trials=25,
          report_test_results=False,
          verbosity=optuna.logging.WARN
        )

        if study.best_value > best_score:
          best_score = study.best_value

        importances = get_importances(best_model.coef_[0], list(X.columns), top=None)
        importances_by_sex_subtype[(sex, subtype)].append(importances)

      print(f"Best score: {best_score}")

  result = defaultdict(dict)
  for key, importances_list in importances_by_sex_subtype.items():
    importances_series: pd.Series = reduce(lambda x, y: x + y, importances_list) / float(len(importances_list))
    importances = [{ "gene": k, "coef": v * v } for k, v in importances_series.items() if v != 0]
    result[key[1]] |= { key[0]: sorted(importances, key=lambda x: x["coef"], reverse=True) }

  return result

In [36]:
result = choose_important_genes(scoring=micro_recall)
open(f"../../preprocessed/{category}/important_genes_logistic_recall.json", "w").write(json.dumps(result))

Finding importances for Male - BCRABL1
Best score: 0.9674418604651163
Finding importances for Male - DUX4IGH
Best score: 0.9906976744186047
Finding importances for Male - ETV6RUNX1
Best score: 0.9860465116279069
Finding importances for Male - PHlike
Best score: 0.9441860465116279
Finding importances for Male - HYPER
Best score: 0.9348837209302326
Finding importances for Male - HYPO
Best score: 0.9534883720930232
Finding importances for Male - KMT2A
Best score: 1.0
Finding importances for Male - iAMP21
Best score: 0.9813953488372092
Finding importances for Male - PAX5
Best score: 0.9488372093023255
Finding importances for Male - TCF3PBX1
Best score: 0.9953488372093023
Finding importances for Female - BCRABL1
Best score: 0.9512012012012011
Finding importances for Female - DUX4IGH
Best score: 0.9945945945945945
Finding importances for Female - ETV6RUNX1
Best score: 1.0
Finding importances for Female - PHlike
Best score: 0.9130630630630632
Finding importances for Female - HYPER
Best score:

104163

Exception ignored in: <function ResourceTracker.__del__ at 0x1109204a0>
Traceback (most recent call last):
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x11244c4a0>
Traceback (most recent call last):
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py",

In [37]:
result = choose_important_genes(scoring=weighted_f1)
open(f"../../preprocessed/{category}/important_genes_logistic_f1.json", "w").write(json.dumps(result))

Finding importances for Male - BCRABL1
Best score: 0.9579332980490743
Finding importances for Male - DUX4IGH
Best score: 0.9906498875801202
Finding importances for Male - ETV6RUNX1
Best score: 0.9861248569502952
Finding importances for Male - PHlike
Best score: 0.9443219160658171
Finding importances for Male - HYPER
Best score: 0.93700388750348
Finding importances for Male - HYPO
Best score: 0.9419255784811439
Finding importances for Male - KMT2A
Best score: 1.0
Finding importances for Male - iAMP21
Best score: 0.973809531657998
Finding importances for Male - PAX5
Best score: 0.9415178529880788
Finding importances for Male - TCF3PBX1
Best score: 0.9898248636233132
Finding importances for Female - BCRABL1
Best score: 0.9428355851258484
Finding importances for Female - DUX4IGH
Best score: 0.9943358743358743
Finding importances for Female - ETV6RUNX1
Best score: 1.0
Finding importances for Female - PHlike
Best score: 0.9107005238152777
Finding importances for Female - HYPER
Best score: 0.

104021

Exception ignored in: <function ResourceTracker.__del__ at 0x1079204a0>
Traceback (most recent call last):
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1100104a0>
Traceback (most recent call last):
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/Users/igor.mandello/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/resource_tracker.py",